Training and validation were performed with the SwissProt Training/Validation dataset used to train the DeepLoc 2.0 server. Testing was performed using the DeepLoc 2.0 Human Protein Atlas (HPA) test set. The data is available at: https://services.healthtech.dtu.dk/services/DeepLoc-2.0/4-Data.php

**References**

Thumuluri, V., Almagro Armenteros, J. J., Johansen, A. R., Nielsen, H. & Winther, O. (2022). *DeepLoc 2.0: multi-label subcellular localization prediction using protein language models.* Nucleic Acids Research, 50(W1), W228–W234. https://doi.org/10.1093/nar/gkac278

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import BinaryCrossentropy

In [2]:
df = pd.read_csv("Swissprot_Train_Validation_dataset.csv")
df.head()

,Unnamed: 0,ACC,Kingdom,Partition,Membrane,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Sequence
0,0,Q28165,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,1,Q86U42,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,2,Q0GA42,Metazoa,3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,3,P82349,Metazoa,1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,4,Q7L5N1,Metazoa,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [3]:
df.shape

(28303, 16)

In [4]:
df = df.dropna(subset = ["Partition", "Sequence", "Nucleus"])
df.shape

(28303, 16)

In [5]:
df = df[["Partition", "Nucleus", "Sequence"]]
df.head()

,Partition,Nucleus,Sequence
0,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [6]:
df_test = pd.read_csv("hpa_testset.csv")
df_test.head()

,sid,Cell membrane,Cytoplasm,Endoplasmic reticulum,Golgi apparatus,Lysosome/Vacuole,Mitochondrion,Nucleus,Peroxisome,Lengths,fasta
0,ENSP00000355743,0,0,0,0,0,0,1,0,496,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,ENSP00000429628,0,0,0,0,0,0,1,0,173,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,ENSP00000266544,0,0,0,0,0,1,0,0,377,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,ENSP00000470652,0,0,0,1,0,0,0,0,410,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,ENSP00000460751,0,0,0,0,0,0,1,0,163,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [7]:
df_test.shape

(1717, 11)

In [8]:
df_test.rename(columns={"fasta": "Sequence"}, inplace=True)
df_test = df_test.dropna(subset = ["Sequence", "Nucleus"])

In [9]:
df_test = df_test[["Nucleus", "Sequence"]]
df_test.head()

,Nucleus,Sequence
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [10]:
#feature engineering - dipeptide composition (as a fraction of number of dipeptides, which is = (sequence length) - 1 )
#dipeptide composition includes some positional information: which amino acids often occur adjacent to each other

amino_acids = "ACDEFGHIKLMNPQRSTVWY"

aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

def dipeptide_comp(sequence):
    # Convert valid amino acids to indices
    indices = np.array([aa_to_idx.get(aa, -1) for aa in sequence])

    first = indices[:-1]
    second = indices[1:]

    # Keep only pairs residues are standard amino acids (not X, B, U)
    valid = (first >= 0) & (second >= 0)

    first = first[valid]
    second = second[valid]

    # Encode each dipeptide as an integer 0–399
    dipeptide_indices = first * 20 + second

    counts = np.bincount(dipeptide_indices, minlength=400)

    # Normalise by number of valid dipeptides
    n_valid = len(dipeptide_indices)

    if n_valid == 0:
        return np.zeros(400)

    return counts / n_valid

In [11]:
df["input_vectors"] = df["Sequence"].apply(dipeptide_comp)
df.head()

,Partition,Nucleus,Sequence,input_vectors
0,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...,"[0.03278688524590164, 0.0, 0.00327868852459016..."
1,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...,"[0.03278688524590164, 0.0, 0.00327868852459016..."
2,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...,"[0.022105263157894735, 0.002105263157894737, 0..."
3,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...,"[0.025078369905956112, 0.0, 0.0031347962382445..."
4,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...,"[0.027607361963190184, 0.003067484662576687, 0..."


In [12]:
df_test["input_vectors"] = df_test["Sequence"].apply(dipeptide_comp)
df_test.head()

,Nucleus,Sequence,input_vectors
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...,"[0.014141414141414142, 0.0, 0.0, 0.00808080808..."
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...,"[0.023255813953488372, 0.0, 0.0058139534883720..."
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...,"[0.007978723404255319, 0.0, 0.0, 0.00265957446..."
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...,"[0.014669926650366748, 0.0, 0.0024449877750611..."
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...,"[0.024691358024691357, 0.006172839506172839, 0..."


In [13]:
#5-fold cross-validation across homology-partitioned training datasets
#first fold of 5-fold cross-validation will use homology partition 0 as the validation set
train_df = df[df["Partition"].isin([1, 2, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 0]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_test = np.stack(df_test["input_vectors"].values)
y_test = np.stack(df_test["Nucleus"].values)

In [14]:
#feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

In [15]:
model = Sequential([
    Dense(units = 128, activation = 'relu', kernel_regularizer = tf.keras.regularizers.l2(0.01)),
    Dense(units = 64, activation = 'relu', kernel_regularizer = tf.keras.regularizers.l2(0.01)),
    Dense(units = 1, activation = 'sigmoid', kernel_regularizer = tf.keras.regularizers.l2(0.01))
])

model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01), loss = BinaryCrossentropy(), metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ])

model.fit(X_train_scaled, y_train, epochs = 50)

Epoch 1/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.7106 - loss: 0.7348 - precision: 0.6246 - recall: 0.4417
Epoch 2/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7217 - loss: 0.6012 - precision: 0.6419 - recall: 0.4701
Epoch 3/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7205 - loss: 0.5910 - precision: 0.6409 - recall: 0.4648
Epoch 4/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7223 - loss: 0.5879 - precision: 0.6450 - recall: 0.4664
Epoch 5/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7222 - loss: 0.5874 - precision: 0.6413 - recall: 0.4750
Epoch 6/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7192 - loss: 0.5877 - precision: 0.6343 - recall: 0.4746
Epoch 7/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7189 - loss: 0.5861 - precision: 0.6330 - recall: 0.4760
Epoch 8/50
699/699 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7232 - loss: 0.5864 - precision: 0.6408 - recall: 0.4827
Epoch 9/50
699/6

In [16]:
results = model.evaluate(X_train_scaled, y_train, return_dict=True)

print(results)

699/699 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7303 - loss: 0.5753 - precision: 0.6500 - recall: 0.5039
{'accuracy': 0.7303491234779358, 'loss': 0.5752819776535034, 'precision': 0.6499589085578918, 'recall': 0.5038858652114868}


In [17]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.5
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.73469730001677
              precision    recall  f1-score   support

         0.0       0.78      0.86      0.82      4092
         1.0       0.60      0.46      0.52      1871

    accuracy                           0.73      5963
   macro avg       0.69      0.66      0.67      5963
weighted avg       0.72      0.73      0.72      5963

[[3515  577]
 [1005  866]]


In [18]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.45
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.7224551400301862
              precision    recall  f1-score   support

         0.0       0.79      0.80      0.80      4092
         1.0       0.56      0.54      0.55      1871

    accuracy                           0.72      5963
   macro avg       0.68      0.67      0.68      5963
weighted avg       0.72      0.72      0.72      5963

[[3293  799]
 [ 856 1015]]


In [19]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.40 #gives quite low precision, so lower thresholds will not be tried
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.6937782995136677
              precision    recall  f1-score   support

         0.0       0.84      0.69      0.76      4092
         1.0       0.51      0.71      0.59      1871

    accuracy                           0.69      5963
   macro avg       0.67      0.70      0.67      5963
weighted avg       0.73      0.69      0.70      5963

[[2816 1276]
 [ 550 1321]]


In [20]:
#2nd fold of 5-fold cross-validation will use homology partition 1 as the validation set
train_df = df[df["Partition"].isin([0, 2, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 1]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

model.fit(X_train_scaled, y_train, epochs = 50)

Epoch 1/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7302 - loss: 0.5777 - precision: 0.6448 - recall: 0.4675
Epoch 2/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7305 - loss: 0.5782 - precision: 0.6463 - recall: 0.4659
Epoch 3/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7294 - loss: 0.5775 - precision: 0.6433 - recall: 0.4661
Epoch 4/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7321 - loss: 0.5780 - precision: 0.6468 - recall: 0.4748
Epoch 5/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7313 - loss: 0.5772 - precision: 0.6478 - recall: 0.4674
Epoch 6/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7290 - loss: 0.5780 - precision: 0.6404 - recall: 0.4707
Epoch 7/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7283 - loss: 0.5773 - precision: 0.6408 - recall: 0.4650
Epoch 8/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7304 - loss: 0.5771 - precision: 0.6434 - recall: 0.4726
Epoch 9/50
715/7

In [21]:
results = model.evaluate(X_train_scaled, y_train, return_dict=True)

print(results)

715/715 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7349 - loss: 0.5678 - precision: 0.6514 - recall: 0.4814
{'accuracy': 0.7348591089248657, 'loss': 0.5678497552871704, 'precision': 0.6513522863388062, 'recall': 0.4813581109046936}


In [22]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.5
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

171/171 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.7002384883507613
              precision    recall  f1-score   support

         0.0       0.74      0.83      0.78      3536
         1.0       0.60      0.45      0.52      1915

    accuracy                           0.70      5451
   macro avg       0.67      0.64      0.65      5451
weighted avg       0.69      0.70      0.69      5451

[[2947  589]
 [1045  870]]


In [23]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.45
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Accuracy (cv set): 0.6984039625756742
              precision    recall  f1-score   support

         0.0       0.76      0.78      0.77      3536
         1.0       0.57      0.56      0.56      1915

    accuracy                           0.70      5451
   macro avg       0.67      0.67      0.67      5451
weighted avg       0.70      0.70      0.70      5451

[[2742  794]
 [ 850 1065]]


In [24]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.40
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy (cv set): 0.6967528893780958
              precision    recall  f1-score   support

         0.0       0.79      0.72      0.76      3536
         1.0       0.56      0.65      0.60      1915

    accuracy                           0.70      5451
   macro avg       0.68      0.69      0.68      5451
weighted avg       0.71      0.70      0.70      5451

[[2554  982]
 [ 671 1244]]


In [25]:
#3rd fold of 5-fold cross-validation will use homology partition 2 as the validation set
train_df = df[df["Partition"].isin([0, 1, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 2]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)


model.fit(X_train_scaled, y_train, epochs = 50)

Epoch 1/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7235 - loss: 0.5848 - precision: 0.6373 - recall: 0.4613
Epoch 2/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7211 - loss: 0.5853 - precision: 0.6297 - recall: 0.4655
Epoch 3/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7218 - loss: 0.5855 - precision: 0.6328 - recall: 0.4617
Epoch 4/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7228 - loss: 0.5859 - precision: 0.6345 - recall: 0.4636
Epoch 5/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7229 - loss: 0.5873 - precision: 0.6311 - recall: 0.4743
Epoch 6/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7219 - loss: 0.5856 - precision: 0.6328 - recall: 0.4622
Epoch 7/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7224 - loss: 0.5852 - precision: 0.6301 - recall: 0.4737
Epoch 8/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7219 - loss: 0.5879 - precision: 0.6339 - recall: 0.4592
Epoch 9/50
706/7

In [26]:
results = model.evaluate(X_train_scaled, y_train, return_dict=True)

print(results)

706/706 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7331 - loss: 0.5771 - precision: 0.6562 - recall: 0.4759
{'accuracy': 0.7331206798553467, 'loss': 0.5771031975746155, 'precision': 0.6562223434448242, 'recall': 0.47592759132385254}


In [27]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.5
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.7368696562554528
              precision    recall  f1-score   support

         0.0       0.77      0.86      0.81      3800
         1.0       0.64      0.49      0.56      1931

    accuracy                           0.74      5731
   macro avg       0.71      0.68      0.68      5731
weighted avg       0.73      0.74      0.73      5731

[[3282  518]
 [ 990  941]]


In [28]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.45
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.7333798638980981
              precision    recall  f1-score   support

         0.0       0.79      0.81      0.80      3800
         1.0       0.61      0.59      0.60      1931

    accuracy                           0.73      5731
   macro avg       0.70      0.70      0.70      5731
weighted avg       0.73      0.73      0.73      5731

[[3065  735]
 [ 793 1138]]


In [29]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.40
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.7185482463793404
              precision    recall  f1-score   support

         0.0       0.81      0.75      0.78      3800
         1.0       0.57      0.66      0.61      1931

    accuracy                           0.72      5731
   macro avg       0.69      0.71      0.70      5731
weighted avg       0.73      0.72      0.72      5731

[[2835  965]
 [ 648 1283]]


In [30]:
#4th fold of 5-fold cross-validation will use homology partition 3 as the validation set
train_df = df[df["Partition"].isin([0, 1, 2, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 3]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

model.fit(X_train_scaled, y_train, epochs = 50)

Epoch 1/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7267 - loss: 0.5814 - precision: 0.6368 - recall: 0.4682
Epoch 2/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7268 - loss: 0.5804 - precision: 0.6389 - recall: 0.4630
Epoch 3/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7281 - loss: 0.5797 - precision: 0.6418 - recall: 0.4643
Epoch 4/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7290 - loss: 0.5795 - precision: 0.6418 - recall: 0.4705
Epoch 5/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7292 - loss: 0.5805 - precision: 0.6425 - recall: 0.4696
Epoch 6/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7280 - loss: 0.5803 - precision: 0.6422 - recall: 0.4624
Epoch 7/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7273 - loss: 0.5810 - precision: 0.6413 - recall: 0.4600
Epoch 8/50
707/707 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7278 - loss: 0.5796 - precision: 0.6415 - recall: 0.4634
Epoch 9/50
707/7

In [31]:
results = model.evaluate(X_train_scaled, y_train, return_dict=True)

print(results)

707/707 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7361 - loss: 0.5747 - precision: 0.6347 - recall: 0.5387
{'accuracy': 0.7360994219779968, 'loss': 0.5746505856513977, 'precision': 0.6346739530563354, 'recall': 0.53866046667099}


In [32]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.5
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.7166432584269663
              precision    recall  f1-score   support

         0.0       0.76      0.84      0.79      3710
         1.0       0.62      0.49      0.55      1986

    accuracy                           0.72      5696
   macro avg       0.69      0.67      0.67      5696
weighted avg       0.71      0.72      0.71      5696

[[3099  611]
 [1003  983]]


In [33]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.45
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy (cv set): 0.7192766853932584
              precision    recall  f1-score   support

         0.0       0.79      0.78      0.78      3710
         1.0       0.60      0.60      0.60      1986

    accuracy                           0.72      5696
   macro avg       0.69      0.69      0.69      5696
weighted avg       0.72      0.72      0.72      5696

[[2906  804]
 [ 795 1191]]


In [34]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.40
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy (cv set): 0.6889044943820225
              precision    recall  f1-score   support

         0.0       0.83      0.66      0.73      3710
         1.0       0.54      0.74      0.62      1986

    accuracy                           0.69      5696
   macro avg       0.68      0.70      0.68      5696
weighted avg       0.73      0.69      0.70      5696

[[2450 1260]
 [ 512 1474]]


In [35]:
#5th fold of 5-fold cross-validation will use homology partition 4 as the validation set
train_df = df[df["Partition"].isin([0, 1, 2, 3])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 4]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

model.fit(X_train_scaled, y_train, epochs = 50)

Epoch 1/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7278 - loss: 0.5786 - precision: 0.6342 - recall: 0.4558
Epoch 2/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7255 - loss: 0.5803 - precision: 0.6294 - recall: 0.4527
Epoch 3/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7283 - loss: 0.5784 - precision: 0.6365 - recall: 0.4532
Epoch 4/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7281 - loss: 0.5791 - precision: 0.6363 - recall: 0.4524
Epoch 5/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7283 - loss: 0.5793 - precision: 0.6367 - recall: 0.4529
Epoch 6/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7283 - loss: 0.5781 - precision: 0.6354 - recall: 0.4558
Epoch 7/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7297 - loss: 0.5792 - precision: 0.6425 - recall: 0.4472
Epoch 8/50
714/714 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7300 - loss: 0.5789 - precision: 0.6417 - recall: 0.4514
Epoch 9/50
714/7

In [36]:
results = model.evaluate(X_train_scaled, y_train, return_dict=True)

print(results)

714/714 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7374 - loss: 0.5673 - precision: 0.6699 - recall: 0.4366
{'accuracy': 0.7374458312988281, 'loss': 0.5672984719276428, 'precision': 0.6699203252792358, 'recall': 0.43658316135406494}


In [37]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.5
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy (cv set): 0.7087147564994507
              precision    recall  f1-score   support

         0.0       0.72      0.89      0.79      3445
         1.0       0.67      0.41      0.51      2017

    accuracy                           0.71      5462
   macro avg       0.70      0.65      0.65      5462
weighted avg       0.70      0.71      0.69      5462

[[3049  396]
 [1195  822]]


In [38]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.45
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy (cv set): 0.712925668253387
              precision    recall  f1-score   support

         0.0       0.74      0.83      0.78      3445
         1.0       0.64      0.51      0.57      2017

    accuracy                           0.71      5462
   macro avg       0.69      0.67      0.68      5462
weighted avg       0.71      0.71      0.71      5462

[[2860  585]
 [ 983 1034]]


In [39]:
y_prob = model.predict(X_cv_scaled)
threshold = 0.40
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (cv set):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

171/171 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Accuracy (cv set): 0.7070670084218235
              precision    recall  f1-score   support

         0.0       0.77      0.76      0.77      3445
         1.0       0.60      0.62      0.61      2017

    accuracy                           0.71      5462
   macro avg       0.69      0.69      0.69      5462
weighted avg       0.71      0.71      0.71      5462

[[2621  824]
 [ 776 1241]]


In [40]:
#final model
X_train = np.stack(df["input_vectors"].values)
y_train = np.stack(df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model.fit(X_train_scaled, y_train, epochs = 50)

Epoch 1/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7254 - loss: 0.5815 - precision: 0.6347 - recall: 0.4718
Epoch 2/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7283 - loss: 0.5820 - precision: 0.6422 - recall: 0.4717
Epoch 3/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.7266 - loss: 0.5822 - precision: 0.6398 - recall: 0.4665
Epoch 4/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.7247 - loss: 0.5820 - precision: 0.6345 - recall: 0.4680
Epoch 5/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7232 - loss: 0.5823 - precision: 0.6318 - recall: 0.4650
Epoch 6/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7255 - loss: 0.5823 - precision: 0.6351 - recall: 0.4721
Epoch 7/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7231 - loss: 0.5825 - precision: 0.6309 - recall: 0.4667
Epoch 8/50
885/885 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7265 - loss: 0.5819 - precision: 0.6365 - recall: 0.4748
Epoch 9/50
885/

In [41]:
results = model.evaluate(X_train_scaled, y_train, return_dict=True)

print(results)

885/885 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7340 - loss: 0.5751 - precision: 0.6611 - recall: 0.4622
{'accuracy': 0.7339504361152649, 'loss': 0.5750557780265808, 'precision': 0.6611241698265076, 'recall': 0.46224281191825867}


In [44]:
y_prob = model.predict(X_test_scaled)
threshold = 0.40
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (test set):", accuracy_score(y_test, yhat))
print(classification_report(y_test, yhat))
print(confusion_matrix(y_test, yhat))

54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Accuracy (test set): 0.6051252184041933
              precision    recall  f1-score   support

           0       0.59      0.58      0.58       824
           1       0.62      0.63      0.62       893

    accuracy                           0.61      1717
   macro avg       0.60      0.60      0.60      1717
weighted avg       0.60      0.61      0.60      1717

[[477 347]
 [331 562]]
